# Fanfics Dataset Analysis using Apache Spark

## Overview
- This notebook demonstrates the setup and verification of a Spark environment on the SDSC Expanse cluster,
- followed by loading and validating a large-scale fanfiction dataset.

## Objectives
- Configure SparkSession based on allocated cluster resources
- Load dataset stored on Lustre filesystem
- Verify distributed processing
- Prepare for large-scale data analysis

## Dataset
- Source: https://huggingface.co/datasets/marianna13/fanfics

## Spark Configuration

- The SparkSession is configured based on the allocated resources in the Expanse Jupyter environment.

### Allocated Resources:
- Total Cores: 16
- Total Memory: 128GB

### Configuration Formula:
- Driver Memory: 2GB (fixed)
- Executor Instances: Total Cores - 1 = 15
- Executor Memory: (Total Memory - Driver Memory) / Executor Instances
- = (128GB - 2GB) / 15 ≈ 8GB


- This ensures efficient parallel execution while reserving resources for the driver node.

In [6]:
import requests
import pandas as pd

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.instances", 15) \
    .getOrCreate()

# Display Spark session to confirm initialization
spark

In [10]:
conf = spark.sparkContext.getConf().getAll()

for key, value in conf:
    print(f"{key}: {value}")

spark.app.startTime: 1777156414194
spark.driver.port: 36477
spark.executor.instances: 15
spark.app.submitTime: 1777156414091
spark.executor.memory: 4g
spark.executor.id: driver
spark.app.name: pyspark-shell
spark.driver.extraJavaOptions: -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED

In [11]:
print("Default Parallelism:", spark.sparkContext.defaultParallelism)

Default Parallelism: 16


In [ ]:
import os
import glob

# ----------------------------------------
# Load Dataset from Lustre Filesystem
# ----------------------------------------
# Each member has his own 186 GB corpus copy at <repo>/shared/fanfics/.
# Resolve the path portably so either member's run finds his own copy.

candidates = [
    os.path.join(os.path.dirname(os.getcwd()), "shared", "fanfics"),  # repo-relative when cwd=notebook/
    os.path.join(os.getcwd(), "shared", "fanfics"),                    # repo-relative when cwd=repo root
    "/expanse/lustre/projects/uci157/dpham5/fanfic-spark-analysis/shared/fanfics",  # Derek absolute
    "/expanse/lustre/projects/uci157/mhayeri/fanfics-analysis/shared/fanfics",       # Mustafa absolute
]
data_path = next((p for p in candidates if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(f"None of the candidate corpus paths exist: {candidates}")
print(f"Using data path: {data_path}")

# ----------------------------------------
# Schema heterogeneity across Parquet shards
# ----------------------------------------
# The marianna13/fanfics corpus was produced in heterogeneous batches: most
# shards encode `language` as string ('en', 'fr', ...), but some shards encode
# it as INT32. Spark's default read picks one global schema from the first
# shard alphabetically and fails when it hits a mismatched shard:
#
#   Py4JJavaError: Parquet column cannot be converted in file
#       part-03500-...snappy.parquet. Column: [language],
#       Expected: string, Found: INT32.
#
# Workaround: read each shard's footer, split the file list by `language`
# physical type, load the two subsets, cast the INT32 subset to string, and
# union. This preserves all rows; the numeric language values will be visibly
# distinct in the §3 frequency table and can be normalized downstream in MS3.

from pyspark.sql.functions import col

all_files = sorted(glob.glob(f"{data_path}/*.parquet"))
print(f"Total shards on disk: {len(all_files)}")

# Use pyarrow for fast metadata reads if available; fall back to Spark otherwise.
try:
    import pyarrow.parquet as pq
    def _lang_type(f):
        return str(pq.read_metadata(f).schema.to_arrow_schema().field('language').type)
except ImportError:
    def _lang_type(f):
        return dict(spark.read.parquet(f).dtypes).get('language', 'MISSING')

str_files = []
int_files = []
for i, f in enumerate(all_files):
    t = _lang_type(f)
    if t == 'string':
        str_files.append(f)
    else:
        int_files.append(f)
    if (i + 1) % 200 == 0:
        print(f"  Scanned {i+1}/{len(all_files)} schemas...")

print(f"String-language shards: {len(str_files)}")
print(f"Int-language shards:    {len(int_files)}")

parts = []
if str_files:
    parts.append(spark.read.parquet(*str_files))
if int_files:
    parts.append(
        spark.read.parquet(*int_files).withColumn("language", col("language").cast("string"))
    )

df = parts[0]
for p in parts[1:]:
    df = df.unionByName(p)

print("Dataset successfully loaded with normalized language schema.")

In [15]:
# ----------------------------------------
# Trigger Spark Job to Validate Data Loading
# ----------------------------------------

row_count = df.count()
print(f"Total number of observations: {row_count}")

Total number of observations: 5252058


In [16]:
# ----------------------------------------
# Inspect Dataset Schema
# ----------------------------------------

df.printSchema()

root
 |-- __null_dask_index__: long (nullable = true)
 |-- TEXT: string (nullable = true)
 |-- CATEGORY: string (nullable = true)
 |-- SOURCE: string (nullable = true)
 |-- language: string (nullable = true)
 |-- text_len: long (nullable = true)
 |-- perplexity_score: double (nullable = true)



In [19]:
# ----------------------------------------
# Display Sample Rows
# ----------------------------------------

# df.show(5, truncate=False)

In [20]:
# ----------------------------------------
# Check Number of Partitions
# ----------------------------------------

num_partitions = df.rdd.getNumPartitions()

print("Number of partitions:", num_partitions)

Number of partitions: 1932


## Section 3 — Data Exploration

All exploration uses Spark DataFrames per the assignment requirement. Plots in §4 use sampled or aggregated results converted to Pandas after the Spark stages.

### 3a. Total observation count

`df.count()` is shown above (5,252,058 rows).

### 3b. Column descriptions and distributions

Numeric columns (`text_len`, `perplexity_score`) get `describe()`. Categorical columns (`CATEGORY`, `SOURCE`, `language`) get frequency tables and distinct counts. The `__null_dask_index__` column is a Dask leftover and will be dropped in preprocessing.

In [ ]:
df.describe(["text_len", "perplexity_score"]).show()

In [ ]:
from pyspark.sql import functions as F

df.groupBy('language').count().orderBy(F.desc('count')).show(30, truncate=False)
print("Distinct languages:", df.select('language').distinct().count())

In [ ]:
df.groupBy('SOURCE').count().orderBy(F.desc('count')).show(10, truncate=False)
print("Distinct SOURCE values:", df.select('SOURCE').distinct().count())

In [ ]:
df.groupBy('CATEGORY').count().orderBy(F.desc('count')).show(50, truncate=False)
print("Approx distinct CATEGORY values:",
      df.agg(F.approx_count_distinct('CATEGORY')).collect()[0][0])

In [ ]:
quantiles_text_len = df.approxQuantile("text_len",
    [0.0, 0.01, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0], 0.001)
quantiles_perp = df.approxQuantile("perplexity_score",
    [0.0, 0.01, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0], 0.001)

print("text_len quantiles:        ", quantiles_text_len)
print("perplexity_score quantiles:", quantiles_perp)

### 3c. Missing and duplicate values

Nulls are counted per column with a single aggregation. Duplicates are defined by exact `TEXT` equality — the practical definition for a text corpus where the same prose may appear under multiple `CATEGORY` rows.

In [ ]:
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show(truncate=False)

In [ ]:
total = df.count()
unique_texts = df.dropDuplicates(['TEXT']).count()
print(f"Total rows:           {total:,}")
print(f"Unique TEXT rows:     {unique_texts:,}")
print(f"Duplicate TEXT rows:  {total - unique_texts:,}")

## Spark UI Verification

A Spark UI screenshot is included showing:
- Multiple executors active
- Tasks distributed across nodes

This confirms that the dataset is processed in a distributed manner.



In [7]:
sc = spark.sparkContext

url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

response = requests.get(url)
executors = response.json()

# Convert to DataFrame
exec_df = pd.DataFrame(executors)[
    ['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']
]

# Convert memory to GB
exec_df['maxMemory_GB'] = (exec_df['maxMemory'] / (1024**3)).round(2)

exec_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,16,1099746508,0,True,1.02
